## GRNBoost2: Gene Regulatory Networks

### 1.1 GRNBoost for network

In [41]:
import pandas as pd
import numpy as np
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2

if __name__ == '__main__':
    # ex_matrix is a DataFrame with gene names as column names
    # ex_matrix = pd.read_csv(<ex_path>, sep='\t')
    exp = pd.read_csv("./MDIC3/exp.txt", sep="\t", index_col=0)
    # transpose GRNBoost format：cells × genes
    ex_matrix = exp.T

    # tf_names is read using a utility function included in Arboreto
    # tf_names = load_tf_names(<tf_path>)
    tf_names = list(ex_matrix.columns)

    network = grnboost2(expression_data=ex_matrix,
                        tf_names=tf_names)

    # network.to_csv('output.tsv', sep='\t', index=False, header=False)
    print(network.shape)

/share/home/zhangze/anaconda3/envs/mdic3/lib/python3.9/site-packages/distributed/client.py:3162: UserWarning: Sending large graph of size 275.02 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


(299675, 3)


In [42]:
# type(network)
# pandas.core.frame.DataFrame
network.head()

,TF,target,importance
4270,gene-evm.model.ptg000012l.209-1,OG0000713,15.617760
4346,gene-evm.model.ptg000013l.789-1,OG0000557,11.031498
4449,gene-evm.model.ptg000018l.352-1,OG0004790,10.865602
4046,gene-evm.model.ptg000008l.306-1,OG0004878,9.931999
4293,gene-evm.model.ptg000012l.67-1,OG0003779,9.628363


### 1.2 network for GRN 

In [43]:
# gene sort for MDIC3
genes = exp.index.tolist()
n_genes = len(genes)
# gene -> index mapping
gene2idx = {g: i for i, g in enumerate(genes)}
print("Number of genes:", n_genes)

# network: DataFrame with columns ['TF', 'target', 'importance']
GRN = np.zeros((n_genes, n_genes), dtype=np.float32)
miss_tf = 0
miss_tg = 0

for tf, target, weight in network[["TF", "target", "importance"]].itertuples(index=False):
    if tf not in gene2idx:
        miss_tf += 1
        continue
    if target not in gene2idx:
        miss_tg += 1
        continue

    i = gene2idx[tf]
    j = gene2idx[target]

    GRN[i, j] = weight

print("TF not found in exp genes:", miss_tf)
print("Target not found in exp genes:", miss_tg)

GRN

Number of genes: 5000
TF not found in exp genes: 0
Target not found in exp genes: 0


array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.34057418, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], dtype=float32)

In [44]:
np.savetxt("./MDIC3/GRN.txt", GRN, fmt="%.6e")

## MDIC3: Cell-Cell Communication

### 2.1 MDIC3 for ccc_adjacency and type_adjacency

In [45]:
import os
import numpy as np
import pandas as pd
import scipy, statsmodels
print("scipy:", scipy.__version__)
print("statsmodels:", statsmodels.__version__)

scipy: 1.13.1
statsmodels: 0.14.6


In [46]:
from MDIC3 import lucky
if __name__ == '__main__':

    AA, gene_exp, cellname = lucky.readexp('./MDIC3/exp.txt')
    labels, label_index, label_cell = lucky.readlabel('./MDIC3/metadata.txt')

    # import the GRN calculated by other methods or tools
    GRN = np.loadtxt('./MDIC3/GRN.txt')

    old_cwd = os.getcwd()
    os.chdir('./MDIC3')

    try:
        # Infer the cell-cell communication
        ccc_adjacency, type_adjacency = lucky.MDIC3_score(AA, GRN, labels, label_index)
        # User can save the inferred results using the following command
        lucky.MDIC3_scoresave(ccc_adjacency, type_adjacency, labels)
    finally:
        # Restore original working directory
        os.chdir(old_cwd) 

Wed Dec 17 15:54:08 2025: Start calculating cell-cell communication.
Wed Dec 17 15:56:08 2025: Complete the cell-cell communication calculation.
Saving cell-cell communication results...


In [47]:
ccc_adjacency

array([[-4.69948106e-03,  8.68234782e-05,  1.70034027e-03, ...,
         9.55403951e-03,  8.36473962e-03,  1.07789149e-02],
       [ 5.94893462e-02,  4.43516886e-03,  1.00708981e-02, ...,
         9.69671342e-03,  7.22650650e-02, -5.98926829e-02],
       [ 1.03447359e-02,  2.32400411e-03,  3.75648744e-03, ...,
        -1.13072884e-02, -2.98076672e-02, -2.25013264e-02],
       ...,
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [48]:
type_adjacency

array([[5.92948233, 6.14835913, 6.10984984],
       [6.14016526, 6.35250537, 6.32374431],
       [6.10321812, 6.3265572 , 6.28359911]])

### 2.2 ccc_adjacency for filtering edges

In [56]:
ccc_adjacency.shape

(7146, 7146)

In [57]:
# all non-zero edges
rows, cols = np.nonzero(ccc_adjacency)
weights = ccc_adjacency[rows, cols]

edge_df = pd.DataFrame({
    "source": rows,
    "target": cols,
    "weight": weights,
    "abs_weight": np.abs(weights)
})

# |weight| descent, top 2000
edge_df = (
    edge_df
    .sort_values("abs_weight", ascending=False)
    .head(2000)
    .drop(columns="abs_weight")
    .reset_index(drop=True)
)

print("Number of edges selected:", edge_df.shape[0])
edge_df

Number of edges selected: 2000


,source,target,weight
0,4981,2326,21.727838
1,4981,577,21.023826
2,4981,6608,20.233483
3,4981,5460,-20.148922
4,4821,577,-19.826140
...,...,...,...
1995,4981,4290,-10.550266
1996,4821,5656,10.550034
1997,4854,2302,10.549288
1998,4771,3158,10.547492


In [58]:
# read metadata.txt
meta = pd.read_csv(
    "./MDIC3/metadata.txt",
    sep="\t",
    header=None,
    names=["spot", "cluster_annos"]
)

# spot sort = ccc_adjacency sort
meta["spot_index"] = np.arange(meta.shape[0])
node_df = meta[["spot_index", "spot", "cluster_annos"]]
node_df


,spot_index,spot,cluster_annos
0,0,8160437867600,Neural-related
1,1,8160437867650,Neural-related
2,2,8160437868000,Nematocyte-related
3,3,8160437869200,Neural-related
4,4,8160437870500,Epidermal/Muscle-related
...,...,...,...
7141,7141,83322365552350,Neural-related
7142,7142,83322365552400,Nematocyte-related
7143,7143,83322365552450,Nematocyte-related
7144,7144,83322365552500,Nematocyte-related


In [59]:
# add source / target typese for dge_df
edge_df = edge_df.merge(
    node_df[["spot_index", "cluster_annos"]],
    left_on="source",
    right_on="spot_index",
    how="left"
).rename(columns={"cluster_annos": "source_type"}).drop(columns="spot_index")

edge_df = edge_df.merge(
    node_df[["spot_index", "cluster_annos"]],
    left_on="target",
    right_on="spot_index",
    how="left"
).rename(columns={"cluster_annos": "target_type"}).drop(columns="spot_index")

edge_df

,source,target,weight,source_type,target_type
0,4981,2326,21.727838,Nematocyte-related,Epidermal/Muscle-related
1,4981,577,21.023826,Nematocyte-related,Epidermal/Muscle-related
2,4981,6608,20.233483,Nematocyte-related,Epidermal/Muscle-related
3,4981,5460,-20.148922,Nematocyte-related,Nematocyte-related
4,4821,577,-19.826140,Epidermal/Muscle-related,Epidermal/Muscle-related
...,...,...,...,...,...
1995,4981,4290,-10.550266,Nematocyte-related,Nematocyte-related
1996,4821,5656,10.550034,Epidermal/Muscle-related,Nematocyte-related
1997,4854,2302,10.549288,Nematocyte-related,Nematocyte-related
1998,4771,3158,10.547492,Nematocyte-related,Neural-related


In [60]:
edge_df.to_csv("./MDIC3/Aurelia.MDIC3.cccTop2000edges.csv", index=None)